In [ ]:
#   numpy(넘파이)는 숫자 여러 개를 '배열(array)'로 묶어 한 번에 계산하게 해 주는 라이브러리입니다.
#   예) [160, 170, 180]에 모두 2를 더하는 계산을 한 줄로 할 수 있습니다.
#   이 노트북에서는 데이터 준비/정규화 같은 'NumPy 흐름'에 사용합니다.
import numpy as np

#   torch는 PyTorch 라이브러리입니다.
#   이번 노트북에서는 다음을 쓰기 위해 사용합니다.
#       (1) torch.nn.Module : PyTorch 모델을 만들기 위한 기본 class
#       (2) torch.nn.Linear : H(x)=a·X_norm + b 계산을 대신해 주는 부품(model 안에 넣음)
#       (3) torch.nn.BCELoss: Binary Cross Entropy Cost 계산을 대신해 주는 부품
#       (4) 자동 미분(autograd)과 optimizer(torch.optim.SGD)
#
#   참고: 이 노트북에서는 'import torch.nn as nn' 같은 줄임 import를 쓰지 않습니다.
#       항상 torch.nn.Linear, torch.nn.BCELoss 처럼 전체 이름을 그대로 적습니다.
#       그래야 이 기능들이 torch.nn 안에 있는 API라는 사실이 코드에 그대로 보입니다.
import torch

In [ ]:
#   ========================================
#   1. 입력값 X와 정답 y 준비   (이전 파일과 동일)
#   ========================================

#   X: 입력값입니다. 여기서는 사람의 키(cm)를 사용합니다.
#       np.array([...]) 는 여러 숫자를 하나의 NumPy 배열로 묶는 것입니다.
X = np.array([160, 170, 180, 190])

#   y: 정답값입니다. 0은 농구선수 아님, 1은 농구선수입니다.
#       X와 y는 순서대로 짝지어집니다. 즉 키 160 → 정답 0, 키 190 → 정답 1.
y = np.array([0, 0, 1, 1])

print('입력값 X: ', X)
print('입력값 y: ', y)

In [ ]:
#   ========================================
#   2. 입력값 정규화    (이전 파일과 동일)
#   ========================================

#   평균(mean)과 표준편차(std)를 계산합니다.
#   - 평균      : 데이터의 중심(가운데쯤 되는 값)
#   - 표준편차   : 데이터가 평균에서 얼마나 넓게 퍼져 있는지를 나타내는 값
X_mean = np.mean(X)
X_std = np.std(X)

#   정규화 공식: (원본값 - 평균) / 표준편차
#   주의: 실제 학습에는 원래 키 X가 아니라, 정규하된 입력값 X_norm을 사용합니다.
#         (X_mean, X_std는 나중에 '새 입력값 예측'에서도 똑같이 다시 씁니다.)
X_norm = (X - X_mean) / X_std

print('입력값 평균 X_mean: ', X_mean)
print('입력값 표준편차 X_std: ', X_std)
print('정규화된 입력값 X_norm: ', X_norm)

In [ ]:
#   ========================================
#   2-1. X_norm 과 t를 PyTorch tensor 로 변환하고 shape을 (n, 1)로 정리
#   ========================================

#   dtype = torch.float32: 소수점 계산(미분)을 위해 실수(float) 형식으로 만듭니다.
X_norm_tensor = torch.tensor(X_norm, dtype = torch.float32)
y_tensor = torch.tensor(y, dtype = torch.float32)

#   torch.nn.Linear(1, 1)에 넣으려면 각 데이터가 '입력 특성 1개'를 가진 형태,
#   즉 shape (n, 1) 이어야 합니다. 그래서 reshape(-1, 1)로 모양을 바꿉니다.
#       -1: 행 개수는 알아서 (여기서는 4)
#        1: 열 개수는 1 (입력 특성 1개)
X_norm_tensor = X_norm_tensor.reshape(-1, 1)
y_tensor = y_tensor.reshape(-1, 1)

print('학습용 입력 tensor X_norm_tensor:\n', X_norm_tensor)
print('학습용 정답 tensor y_tensor:\n', y_tensor)

#   shape을 꼭 확인합니다. 둘 다 (4, 1) 이어야 합니다.
print('X_norm_tensor shape: ', X_norm_tensor.shape)
print('y_tensor shape: ', y_tensor.shape)

In [ ]:
#   ========================================
#   3. PerceptronModel 정의 (torch.nn.Module 상속)
#   ========================================

#   torch.nn.Module을 상속받아 우리만의 모델 class를 만듭니다.
#   이전 파일에서 흩어져 있던 linear 생성과 H/z 계산을 이 안으로 모읍니다.
class PerceptronModel(torch.nn.Module):
    
    def __init__(self):
        #   super().__init__(): 부모 class인 torch.nn.Module의 초기화를 먼저 실행합니다.
        #                       이 줄이 있어야 PyTorch가 self.linear의 weight, bias를
        #                       '학습 대상 파라미터'로 제대로 등록하고 관리합니다. (반드시 필요)
        super().__init__()
        
        #   self.linear        : 이 모델이 가지고 있는 선형 계산 부품입니다.
        #                        torch.nn.Linear(1, 1)은 입력 특성 1개를 받아 출력 1개(H 값)를 내보냅니다.
        #                        내부적으로 H(x)=a·X_norm + b를 계산하며,
        #                        기존 강의의 a는 self.linear.weight, b는 self.linear.bias 입니다.
        self.linear = torch.nn.Linear(1, 1)
        
    def forward(self, x):
        #   forward             : 입력 x가 들어왔을 때 실제 계산 순서를 정의하는 함수입니다.
        #                         이전 파일에서 학습 루프에 직접 적었던 두 줄이 바로 여기로 들어왔습니다.
        
        #   H(x) = a·X_norm + b     (sigmoid 전의 선형 계산값 - 확률이 아닙니다.)
        H = self.linear(x)
        
        # z= sigmoid(H)             (0~1 사이의 예측 확률)
        z = torch.sigmoid(H)
        
        #   이번 파일은 torch.nn.BCELoss()를 쓰므로, H가 아니라 z를 반환합니다.
        return z